# 02 — Supervised Baseline: ResNet18 on AI vs Human art

**Goal**: train the simplest possible supervised classifier and establish a fixed reference point.
All subsequent notebooks (augmentation study, SimCLR, pseudo-labeling) are compared against this run.

Figures → `reports/figures/02_supervised_baseline/`  
CSV    → `reports/02_baseline_results.csv`

## 0. Setup & training

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not ((PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'datasets').exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'src').exists() or not (PROJECT_ROOT / 'datasets').exists():
    raise RuntimeError('No project root found. Please run this script from within the project directory or one of its subdirectories.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as T

from src.train_supervised import train_supervised
from src.augmentations import get_train_transform, list_augmentation_names
from src.utils import (
    fig_path, save_figure, save_results_csv,
    compute_metrics_per_class, plot_confusion_matrix,
    plot_roc_curve, plot_pr_curve, plot_training_curves,
)

REPORTS_DIR = str(PROJECT_ROOT / 'reports')
NB          = '02_supervised_baseline'

%matplotlib inline

## 0b. Augmentation visual preview

Pick one sample image and show how each augmentation pipeline transforms it.

In [ ]:
import random

# --- pick one sample image from the dataset root ---
data_root = PROJECT_ROOT / 'datasets' / 'raw'
all_imgs  = sorted(data_root.rglob('*.jpg')) + sorted(data_root.rglob('*.png'))
if not all_imgs:
    raise FileNotFoundError(f'No images found in {data_root}')
sample_path = all_imgs[len(all_imgs) // 2]  # middle of sorted list — deterministic
base_img    = Image.open(sample_path).convert('RGB').resize((224, 224))

aug_names = list_augmentation_names()
n_cols    = len(aug_names) + 1  # +1 for the original

fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5))

# original
axes[0].imshow(base_img)
axes[0].set_title('original', fontsize=8, fontweight='bold')
axes[0].axis('off')

# one transformed version per augmentation pipeline
for ax, aug_name in zip(axes[1:], aug_names):
    transform = get_train_transform(aug_name, image_size=224)
    # apply and convert tensor → PIL for display
    t = transform(base_img)  # Tensor C×H×W in [-1,1] or [0,1]
    # undo normalise: clamp to [0,1] for display
    t_disp = t.clone()
    if t_disp.min() < 0:
        # assume ImageNet normalisation: mean=[0.485,0.456,0.406] std=[0.229,0.224,0.225]
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        t_disp = t_disp * std + mean
    t_disp = t_disp.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(t_disp)
    ax.set_title(aug_name, fontsize=7)
    ax.axis('off')

fig.suptitle('Augmentation visual preview — same image, all pipelines', fontsize=10)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'aug_preview.png'), show=True)

In [ ]:
config = {
    'data_root':         str(PROJECT_ROOT / 'datasets' / 'raw'),
    'model_name':        'resnet18',
    'augmentation_name': 'weak',
    'num_epochs':        8,   # best value found
    'batch_size':        32,  # for a fair comparison with ViT, we keep the same batch size
    'image_size':        224,
    'num_workers':       4,
    'split_ratios':      (0.7, 0.15, 0.15),
    'seed':              42,
    'checkpoint_dir':    str(PROJECT_ROOT / 'checkpoints'),
    'device':            'cuda' if torch.cuda.is_available() else 'cpu',
}

results = train_supervised(config)
print(results)

## 1. Training curves

In [ ]:
fig = plot_training_curves(
    results['history'],
    path=fig_path(REPORTS_DIR, NB, 'training_curves.png'),
    show=True,
)

## 2. Test set metrics

In [ ]:
m = results['test_metrics_full']

print('=== Test set results =================================')
print(f"  Accuracy          : {m['accuracy']:.4f}")
print(f"  Balanced Accuracy  : {m['balanced_accuracy']:.4f}")
print(f"  F1                : {m['f1']:.4f}")
print(f"  ROC-AUC           : {m['auc']:.4f}")
print(f"  Avg Precision (AP): {m['avg_precision']:.4f}")
print(f"  MCC               : {m['mcc']:.4f}")
print(f"  Specificity       : {m['specificity']:.4f}")
print(f"  Precision         : {m['precision']:.4f}")
print(f"  Recall            : {m['recall']:.4f}")
print('------------------------------------------------------')
print(f"  TP={m['tp']}  FP={m['fp']}  FN={m['fn']}  TN={m['tn']}")

## 3. Per-class breakdown

In [ ]:
y_true = results['test_y_true']
y_pred = results['test_y_pred']
y_prob = results['test_y_prob']

df_per_class = compute_metrics_per_class(y_true, y_pred)
display(df_per_class.style.format({
    'precision': '{:.3f}', 'recall': '{:.3f}',
    'f1-score':  '{:.3f}', 'support': '{:.0f}',
}))

## 4. Confusion matrix

In [ ]:
plot_confusion_matrix(
    y_true, y_pred,
    title='Confusion Matrix — ResNet18 weak aug (test set)',
    path=fig_path(REPORTS_DIR, NB, 'confusion_matrix.png'),
    show=True,
)

## 5. ROC and PR curves

In [ ]:
plot_roc_curve(
    y_true, y_prob, label='ResNet18 weak',
    path=fig_path(REPORTS_DIR, NB, 'roc_curve.png'),
    show=True,
)

plot_pr_curve(
    y_true, y_prob, label='ResNet18 weak',
    path=fig_path(REPORTS_DIR, NB, 'pr_curve.png'),
    show=True,
)

## 6. Error analysis — false positives & false negatives

In [ ]:
test_paths = results['test_paths']

y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)
y_prob_arr = np.array(y_prob)

fp_idx = np.where((y_pred_arr == 1) & (y_true_arr == 0))[0]
fn_idx = np.where((y_pred_arr == 0) & (y_true_arr == 1))[0]

def show_errors(indices, title, n=4):
    idxs = indices[:n]
    fig, axes = plt.subplots(1, len(idxs), figsize=(3*len(idxs), 3))
    if len(idxs) == 1: axes = [axes]
    for ax, i in zip(axes, idxs):
        ax.imshow(Image.open(test_paths[i]).convert('RGB'))
        ax.axis('off')
        label_str = 'Real' if y_true_arr[i] == 0 else 'AI'
        pred_str  = 'Real' if y_pred_arr[i] == 0 else 'AI'
        ax.set_title(f'GT:{label_str}\nPred:{pred_str}\np={y_prob_arr[i]:.2f}', fontsize=8)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return fig

fig_fp = show_errors(fp_idx, 'False Positives — Real classified as AI')
save_figure(fig_fp, fig_path(REPORTS_DIR, NB, 'false_positives.png'), show=True)

fig_fn = show_errors(fn_idx, 'False Negatives — AI classified as Real')
save_figure(fig_fn, fig_path(REPORTS_DIR, NB, 'false_negatives.png'), show=True)

## 7. Save ResNet18 results to CSV

In [ ]:
epoch_records = results['history'].copy()
test_record   = {'epoch': 'test', **results['test_metrics_full'],
                 'augmentation': config['augmentation_name'],
                 'model': config['model_name'], 'seed': config['seed']}

save_results_csv(
    epoch_records,
    str(Path(REPORTS_DIR) / '02_baseline_history.csv'),
)
save_results_csv(
    [test_record],
    str(Path(REPORTS_DIR) / '02_baseline_results.csv'),
)
print(f"Done. Figures → reports/figures/{NB}/")
print( 'CSV   → reports/02_baseline_history.csv')
print( 'CSV   → reports/02_baseline_results.csv')

---
## 8. ViT-B/16 supervised baseline
Motivation: ResNet18 has strong local inductive biases (3×3 convolution, translational equivariance) that favor the recognition of textures and local artifacts typical of AI-generated images. ViT-B/16, on the contrary, models global relationships through self-attention and has no spatial inductive biases; for this reason it is more data-hungry but can capture long-range structural/compositional patterns.

In [ ]:
config_vit = {
    'data_root':         str(PROJECT_ROOT / 'datasets' / 'raw'),
    'model_name':        'vit_b_16',   
    'augmentation_name': 'weak',
    'num_epochs':        20,           
    'batch_size':        32,           
    'image_size':        224,
    'num_workers':       4,
    'split_ratios':      (0.7, 0.15, 0.15),
    'seed':              42,           
    'checkpoint_dir':    str(PROJECT_ROOT / 'checkpoints'),
    'device':            'cuda' if torch.cuda.is_available() else 'cpu',
}

print(f"[ViT] Running on: {config_vit['device']}")
results_vit = train_supervised(config_vit)
print(results_vit)

## 9. ViT — Training curves & test metrics

In [ ]:
# --- training curves ---
fig_vit_curves = plot_training_curves(
    results_vit['history'],
    path=fig_path(REPORTS_DIR, NB, 'vit_training_curves.png'),
    show=True,
)

# --- confusion matrix ---
y_true_vit = results_vit['test_y_true']
y_pred_vit = results_vit['test_y_pred']
y_prob_vit = results_vit['test_y_prob']

plot_confusion_matrix(
    y_true_vit, y_pred_vit,
    title='Confusion Matrix — ViT-B/16 weak aug (test set)',
    path=fig_path(REPORTS_DIR, NB, 'vit_confusion_matrix.png'),
    show=True,
)

# --- test metrics printout ---
mv = results_vit['test_metrics_full']
print('=== ViT-B/16 Test set results ========================')
print(f"  Accuracy          : {mv['accuracy']:.4f}")
print(f"  Balanced Accuracy  : {mv['balanced_accuracy']:.4f}")
print(f"  F1                : {mv['f1']:.4f}")
print(f"  ROC-AUC           : {mv['auc']:.4f}")
print(f"  Avg Precision (AP): {mv['avg_precision']:.4f}")
print(f"  MCC               : {mv['mcc']:.4f}")
print('------------------------------------------------------')
print(f"  TP={mv['tp']}  FP={mv['fp']}  FN={mv['fn']}  TN={mv['tn']}")

## 10. ResNet18 vs ViT-B/16

In [ ]:
# ── 10a. Summary table ──────────────────────────────────────────────────────
m_rn  = results['test_metrics_full']
m_vit = results_vit['test_metrics_full']

METRICS = ['accuracy', 'balanced_accuracy', 'f1', 'auc', 'avg_precision', 'mcc',
           'precision', 'recall', 'specificity']

df_compare = pd.DataFrame({
    'Metric':    [m.replace('_', ' ').title() for m in METRICS],
    'ResNet18':  [round(m_rn[k],  4) for k in METRICS],
    'ViT-B/16':  [round(m_vit[k], 4) for k in METRICS],
})
df_compare['Δ (ViT − RN18)'] = (df_compare['ViT-B/16'] - df_compare['ResNet18']).round(4)
df_compare = df_compare.set_index('Metric')

# colour positive deltas green, negative red
def colour_delta(val):
    colour = 'color: #2a7a2a' if val > 0 else ('color: #c0392b' if val < 0 else '')
    return colour

styled = df_compare.style \
    .format('{:.4f}') \
    .map(colour_delta, subset=['Δ (ViT − RN18)']) \
    .set_caption('ResNet18 vs ViT-B/16 — supervised baseline, same split & augmentation')

display(styled)

# Save as CSV
save_results_csv(
    df_compare.reset_index().to_dict(orient='records'),
    str(Path(REPORTS_DIR) / '02_resnet_vs_vit.csv'),
)
print('CSV saved → reports/02_resnet_vs_vit.csv')

In [ ]:
# ── 10b. Bar chart: key metrics side-by-side ─────────────────────────────────
BAR_METRICS   = ['accuracy', 'f1', 'auc', 'avg_precision', 'mcc']
BAR_LABELS    = ['Accuracy', 'F1', 'ROC-AUC', 'Avg Precision', 'MCC']
rn_vals  = [m_rn[k]  for k in BAR_METRICS]
vit_vals = [m_vit[k] for k in BAR_METRICS]

x      = np.arange(len(BAR_LABELS))
width  = 0.35
colors = {'rn': '#4C72B0', 'vit': '#DD8452'}

fig, ax = plt.subplots(figsize=(9, 4.5))
bars_rn  = ax.bar(x - width/2, rn_vals,  width, label='ResNet18',  color=colors['rn'],  alpha=0.85)
bars_vit = ax.bar(x + width/2, vit_vals, width, label='ViT-B/16',  color=colors['vit'], alpha=0.85)

# value labels on top of each bar
for bar in bars_rn + bars_vit:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(BAR_LABELS)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('ResNet18 vs ViT-B/16 — supervised baseline (weak aug, seed=42)', fontsize=11)
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'resnet_vs_vit_bar.png'), show=True)

In [ ]:
# ── 10c. ROC curves overlaid ─────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc as sk_auc

fpr_rn,  tpr_rn,  _ = roc_curve(results['test_y_true'],     results['test_y_prob'])
fpr_vit, tpr_vit, _ = roc_curve(results_vit['test_y_true'], results_vit['test_y_prob'])

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_rn,  tpr_rn,  lw=2, color=colors['rn'],
        label=f"ResNet18  (AUC={m_rn['auc']:.3f})")
ax.plot(fpr_vit, tpr_vit, lw=2, color=colors['vit'],
        label=f"ViT-B/16  (AUC={m_vit['auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — ResNet18 vs ViT-B/16', fontsize=11)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'resnet_vs_vit_roc.png'), show=True)

In [ ]:
# ── 10d. Training loss overlaid ──────────────────────────────────────────────
hist_rn  = pd.DataFrame(results['history'])
hist_vit = pd.DataFrame(results_vit['history'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Loss
axes[0].plot(hist_rn['epoch'],  hist_rn['train_loss'],  '-o', color=colors['rn'],
             label='ResNet18 train', lw=2, ms=4)
axes[0].plot(hist_rn['epoch'],  hist_rn['val_loss'],    '--o', color=colors['rn'],
             label='ResNet18 val',   lw=2, ms=4, alpha=0.7)
axes[0].plot(hist_vit['epoch'], hist_vit['train_loss'], '-s', color=colors['vit'],
             label='ViT-B/16 train', lw=2, ms=4)
axes[0].plot(hist_vit['epoch'], hist_vit['val_loss'],   '--s', color=colors['vit'],
             label='ViT-B/16 val',   lw=2, ms=4, alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Val Accuracy
axes[1].plot(hist_rn['epoch'],  hist_rn['val_accuracy'],  '-o', color=colors['rn'],
             label='ResNet18',  lw=2, ms=4)
axes[1].plot(hist_vit['epoch'], hist_vit['val_accuracy'], '-s', color=colors['vit'],
             label='ViT-B/16', lw=2, ms=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle('ResNet18 vs ViT-B/16 — learning curves', fontsize=11)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'resnet_vs_vit_curves.png'), show=True)

## 11. Save ViT results to CSV

In [ ]:
vit_epoch_records = results_vit['history'].copy()
vit_test_record   = {'epoch': 'test', **results_vit['test_metrics_full'],
                     'augmentation': config_vit['augmentation_name'],
                     'model': config_vit['model_name'], 'seed': config_vit['seed']}

save_results_csv(
    vit_epoch_records,
    str(Path(REPORTS_DIR) / '02_vit_history.csv'),
)
save_results_csv(
    [vit_test_record],
    str(Path(REPORTS_DIR) / '02_vit_results.csv'),
)

print(f"Done. Figures → reports/figures/{NB}/")
print( 'CSV   → reports/02_vit_history.csv')
print( 'CSV   → reports/02_vit_results.csv')
print( 'CSV   → reports/02_resnet_vs_vit.csv')